# S08 - Creating Databases and Tables

## Libraries

In [1]:
import os
import sqlite3 as sql
import pandas as pd

## Functions

In [2]:
# Function taking a SQL query file to format its text for Jupyter Notebook
def query_text_format(path, file):
    # Change directory
    try:
        os.chdir(path)
    except FileNotFoundError:
        print(f"El directorio {path} no se encontró.")
        return
    
    # Read file
    try:
        with open(file, 'r') as f:
            query_log = f.read()
    except FileNotFoundError:
        print(f"El archivo {file} no se encontró.")
        return
    
    # Text formatting
    # Removal of SQL comment closing
    query_log = query_log.replace(' */', '')
    # Split into lines
    query_log_lines = query_log.split('\n')
    
    # Enumerated iteration over all file lines
    for i, line in enumerate(query_log_lines):
        query_log_lines[i] = line.strip()
        
        # Replacement of SQL comment opening
        if query_log_lines[i][:2] == '/*':
            # Search two integers to detect lesson start
            double_int = []            
            for char in line[3:5]:
                try:
                    if isinstance(int(char), int):
                        double_int.append(1)
                except:
                    pass
            
            # If two integers found, replacement according to lesson start
            if len(double_int) == 2:
                query_log_lines[i] = query_log_lines[i].replace('/*', '###')
            # If a letter, replacement according to assessment exercise start
            elif query_log_lines[i][3].isalpha() and query_log_lines[i][4] == '.':
                query_log_lines[i] = query_log_lines[i].replace('/*', '####')
            # Remaining SQL comments are Python comments, too
            else:
                query_log_lines[i] = query_log_lines[i].replace('/*', '#')
        
        # If line not empty
        elif line != '':
            query_log_lines[i] = '\t' + query_log_lines[i] + ' \\'
        
        # Rest of cases (potential)
        else:
            pass
        
    return '\n'.join(query_log_lines)

In [3]:
# Function taking the SQL queries and building Python code to execute them
def build_python_queries(text):
    # Split into lines
    query_lines = text.split('\n')
    
    # Build SQL queries in Python
    resulting_text = []
    flg_first_sql_line = True
    
    for i, line in enumerate(query_lines):
        # Empty line
        if line == '':
            resulting_text.append(line)
        # Line starting with tab
        elif query_lines[i][0] == '\t':
            # First query line
            if flg_first_sql_line:
                # First query line in one line query
                if (i + 1 <= len(query_lines) - 1) and ((query_lines[i + 1] == '') or ((query_lines[i + 1][0] != '') and (query_lines[i + 1][0] == '#'))):
                    resulting_text.append('query = " \\')
                    resulting_text.append(line)
                    resulting_text.append('\t"')
                    resulting_text.append('execute_query(query, conn)')
                # First query line in multiple lines query
                else:
                    resulting_text.append('query = " \\')
                    resulting_text.append(line)
                    flg_first_sql_line = False
            # Second, or other, query line
            else:
                # Last query line
                if (i + 1 <= len(query_lines) - 1) and ((query_lines[i + 1] == '') or ((query_lines[i + 1][0] != '') and (query_lines[i + 1][0] == '#'))):
                    resulting_text.append(line)
                    resulting_text.append('\t"')
                    resulting_text.append('execute_query(query, conn)')
                    flg_first_sql_line = True
                # Query line, absolute last one
                elif i == len(query_lines) - 1:
                    resulting_text.append(line)
                    resulting_text.append('\t"')
                    resulting_text.append('execute_query(query, conn)')
                # Query lines that are not first nor last
                else:
                    resulting_text.append(line)
        # Other lines
        else:
            resulting_text.append(line)
    
    return '\n'.join(resulting_text)

In [4]:
# Function taking the SQL query text and executing it
def execute_query(query_text, connection):
    try:
        # Check if the query is a SELECT statement
        if query_text.strip().upper().startswith("SELECT"):
            # DataFrame from query
            results_df = pd.read_sql_query(
                query_text,
                connection
            )
            print(results_df)
        else:
            # Execute queries that do not return results
            with connection:
                connection.execute(query_text)
            print("Query executed successfully.")
    except Exception as e:
        print(f"Error executing the query: {e}")

## Settings

In [5]:
# Limit removal for showing pandas.DataFrames' columns
pd.set_option('display.max_columns', None)
# Limit removal for showing pandas.DataFrames' rows
pd.set_option('display.max_rows', None)
# Modification of console with for displaying
pd.set_option('display.width', 8000)

## SQL Query Formatting

In [6]:
# First formatting
path = 'G:\\15_Estudio\\Udemy\\SQL - Portilla Complete Bootcamp'
file = 'UDM-SQL-BTCMP--008.sql'

first_pass = query_text_format(path, file)
print(first_pass)

### 59. Introduction
# Blank/In notes


### 60. Data Types
# Blank/In notes

### 61. Primary Keys and Foreign Keys
# Blank/In notes


### 62. Constraints
# Blank/In notes


### 63. CREATE table
# Create the account table
	CREATE TABLE account( \
	user_id SERIAL PRIMARY KEY, \
	username VARCHAR(50) UNIQUE NOT NULL, \
	password VARCHAR(50) NOT NULL, \
	email VARCHAR(250) UNIQUE NOT NULL, \
	created_on TIMESTAMP NOT NULL, \
	last_login TIMESTAMP \
	); \

# Create the job table
	CREATE TABLE job( \
	job_id SERIAL PRIMARY KEY, \
	job_name VARCHAR(200) UNIQUE NOT NULL \
	); \

# Create intermediary table account <-> job
	CREATE TABLE account_job( \
	user_id INTEGER REFERENCES account(user_id), \
	job_id INTEGER REFERENCES job(job_id), \
	hire_date TIMESTAMP \
	); \


### 64. INSERT
# Explore the account table
	SELECT * \
	FROM account; \

# Explore the job table
	SELECT * \
	FROM job; \

# Explore the account_job table
	SELECT * \
	FROM account_job; \

# Manual row insertion into account
	IN

In [7]:
# Final formatting
working_text = build_python_queries(first_pass)
print(working_text)

### 59. Introduction
# Blank/In notes


### 60. Data Types
# Blank/In notes

### 61. Primary Keys and Foreign Keys
# Blank/In notes


### 62. Constraints
# Blank/In notes


### 63. CREATE table
# Create the account table
query = " \
	CREATE TABLE account( \
	user_id SERIAL PRIMARY KEY, \
	username VARCHAR(50) UNIQUE NOT NULL, \
	password VARCHAR(50) NOT NULL, \
	email VARCHAR(250) UNIQUE NOT NULL, \
	created_on TIMESTAMP NOT NULL, \
	last_login TIMESTAMP \
	); \
	"
execute_query(query, conn)

# Create the job table
query = " \
	CREATE TABLE job( \
	job_id SERIAL PRIMARY KEY, \
	job_name VARCHAR(200) UNIQUE NOT NULL \
	); \
	"
execute_query(query, conn)

# Create intermediary table account <-> job
query = " \
	CREATE TABLE account_job( \
	user_id INTEGER REFERENCES account(user_id), \
	job_id INTEGER REFERENCES job(job_id), \
	hire_date TIMESTAMP \
	); \
	"
execute_query(query, conn)


### 64. INSERT
# Explore the account table
query = " \
	SELECT * \
	FROM account; \
	"
execute_query(q

This text will be used to create the whole of the Practice section in this notebook.

## Practice

### Connection with Data Base

In [8]:
# Connection and cursor
conn = sql.connect('G:\\15_Estudio\\Udemy\\SQL - Portilla Complete Bootcamp\\learning.db')
cur = conn.cursor()

### 59. Introduction

In [9]:
# Blank/In notes

### 60. Data Types

In [10]:
# Blank/In notes

### 61. Primary Keys and Foreign Keys

In [11]:
# Blank/In notes

### 62. Constraints

In [12]:
# Blank/In notes

### 63. CREATE table

In [13]:
# Create the account table
query = " \
	CREATE TABLE account( \
	user_id INTEGER PRIMARY KEY AUTOINCREMENT, \
	username TEXT UNIQUE NOT NULL, \
	password TEXT NOT NULL, \
	email TEXT UNIQUE NOT NULL, \
	created_on DATETIME NOT NULL, \
	last_login DATETIME \
	); \
	"
execute_query(query, conn)
# In Postgre it was:
#	CREATE TABLE account( \
#	user_id SERIAL PRIMARY KEY, \
#	username VARCHAR(50) UNIQUE NOT NULL, \
#	password VARCHAR(50) NOT NULL, \
#	email VARCHAR(250) UNIQUE NOT NULL, \
#	created_on TIMESTAMP NOT NULL, \
#	last_login TIMESTAMP \
#	); \

Query executed successfully.


In [14]:
# Create the job table
query = " \
	CREATE TABLE job( \
	job_id SERIAL PRIMARY KEY, \
	job_name VARCHAR(200) UNIQUE NOT NULL \
	); \
	"
execute_query(query, conn)

Query executed successfully.


In [15]:
# Create intermediary table account <-> job
query = " \
	CREATE TABLE account_job( \
	user_id INTEGER REFERENCES account(user_id), \
	job_id INTEGER REFERENCES job(job_id), \
	hire_date TIMESTAMP \
	); \
	"
execute_query(query, conn)

Query executed successfully.


### 64. INSERT

In [16]:
# Explore the account table
query = " \
	SELECT * \
	FROM account; \
	"
execute_query(query, conn)

Empty DataFrame
Columns: [user_id, username, password, email, created_on, last_login]
Index: []


In [17]:
# Explore the job table
query = " \
	SELECT * \
	FROM job; \
	"
execute_query(query, conn)

Empty DataFrame
Columns: [job_id, job_name]
Index: []


In [18]:
# Explore the account_job table
query = " \
	SELECT * \
	FROM account_job; \
	"
execute_query(query, conn)

Empty DataFrame
Columns: [user_id, job_id, hire_date]
Index: []


In [19]:
# Manual row insertion into account
query = " \
	INSERT INTO account(username, password, email, created_on) \
	VALUES( \
	'Jose', 'password', 'jose@mail.com', CURRENT_TIMESTAMP \
	); \
	"
execute_query(query, conn)

Query executed successfully.


In [20]:
# Manual row insertion into job
query = " \
	INSERT INTO job(job_name) \
	VALUES( \
	'Astronaut' \
	); \
	"
execute_query(query, conn)

Query executed successfully.


In [21]:
query = " \
	INSERT INTO job(job_name) \
	VALUES( \
	'President' \
	); \
	"
execute_query(query, conn)

Query executed successfully.


In [22]:
# Manual row insertion into account_job
query = " \
	INSERT INTO account_job(user_id, job_id, hire_date) \
	VALUES( \
	1, 1, CURRENT_TIMESTAMP \
	); \
	"
execute_query(query, conn)

Query executed successfully.


In [23]:
query = " \
	INSERT INTO account_job(user_id, job_id, hire_date) \
	VALUES( \
	10, 10, CURRENT_TIMESTAMP \
	); \
	"
execute_query(query, conn)
# This should return an error: these foreign keys don't exist on their
# parent tables as primary keys

Query executed successfully.


### 65. UPDATE

In [24]:
# Set values using a condition
query = " \
	UPDATE account \
	SET last_login = CURRENT_TIMESTAMP \
	WHERE last_login IS NULL; \
	"
execute_query(query, conn)

Query executed successfully.


In [25]:
# Reset without condition
query = " \
	UPDATE account \
	SET last_login = CURRENT_TIMESTAMP; \
	"
execute_query(query, conn)

Query executed successfully.


In [26]:
# Based on another column
query = " \
	UPDATE account \
	SET last_login = created_on; \
	"
execute_query(query, conn)

Query executed successfully.


In [27]:
# Explore account, job, account_job tables
query = " \
	SELECT * \
	FROM account; \
	"
execute_query(query, conn)

   user_id username  password          email           created_on           last_login
0        1     Jose  password  jose@mail.com  2025-02-16 17:00:42  2025-02-16 17:00:42


In [28]:
query = " \
	SELECT * \
	FROM job; \
	"
execute_query(query, conn)

  job_id   job_name
0   None  Astronaut
1   None  President


In [29]:
query = " \
	SELECT * \
	FROM account_job; \
	"
execute_query(query, conn)

   user_id  job_id            hire_date
0        1       1  2025-02-16 17:00:43
1       10      10  2025-02-16 17:00:43


In [30]:
# Update from another table
query = " \
	UPDATE account_job \
	SET hire_date = account.created_on \
	FROM account \
	WHERE account_job.user_id = account.user_id; \
	"
execute_query(query, conn)

Query executed successfully.


In [31]:
# Use of RETURNING
query = " \
	UPDATE account \
	SET last_login = CURRENT_TIMESTAMP \
	RETURNING email, created_on, last_login; \
	"
execute_query(query, conn)

Query executed successfully.


### 66. DELETE

In [32]:
# Explore job
query = " \
	SELECT * \
	FROM job; \
	"
execute_query(query, conn)

  job_id   job_name
0   None  Astronaut
1   None  President


In [33]:
# Add a job
query = " \
	INSERT INTO job(job_name) \
	VALUES( \
	'Cowboy' \
	); \
	"
execute_query(query, conn)

Query executed successfully.


In [34]:
# Remove a job
query = " \
	DELETE FROM job \
	WHERE job_name = 'Cowboy' \
	RETURNING job_id, job_name; \
	"
execute_query(query, conn)
# If run again, returns nothing, because entry had been already deleted

Query executed successfully.


### 67. ALTER Table

In [35]:
# Create the information table
query = " \
	CREATE TABLE information( \
	info_id SERIAL PRIMARY KEY, \
	title VARCHAR(500) NOT NULL, \
	person VARCHAR(50) NOT NULL UNIQUE \
	); \
	"
execute_query(query, conn)

Query executed successfully.


In [36]:
# Explore the information table
query = " \
	SELECT * \
	FROM information; \
	"
execute_query(query, conn)

Empty DataFrame
Columns: [info_id, title, person]
Index: []


In [37]:
# Rename this table
query = " \
	ALTER TABLE information \
	RENAME TO new_info; \
	"
execute_query(query, conn)

Query executed successfully.


In [38]:
# Explore the new_info table
query = " \
	SELECT * \
	FROM new_info; \
	"
execute_query(query, conn)

Empty DataFrame
Columns: [info_id, title, person]
Index: []


In [39]:
# Rename a column
query = " \
	ALTER TABLE new_info \
	RENAME COLUMN person TO people; \
	"
execute_query(query, conn)

Query executed successfully.


In [40]:
# Insert rows into the table
query = " \
	INSERT INTO new_info(title) \
	VALUES( \
	'Some New Title' \
	); \
	"
execute_query(query, conn)
# Error raised because of null value in people column

Error executing the query: NOT NULL constraint failed: new_info.people


In [41]:
# The following query in Postgre:
#	ALTER TABLE new_info \
#	ALTER COLUMN people DROP NOT NULL; \

# Can't be executed like that in SQLite, needing a series
# of steps that include creating a new table, like this:

# Step 1: Create a new table with the desired structure
create_query = " \
    CREATE TABLE new_info_temp ( \
        info_id INTEGER PRIMARY KEY AUTOINCREMENT, \
        title TEXT NOT NULL, \
        people TEXT UNIQUE  \
    ); \
"
execute_query(create_query, conn)

# Step 2: Copy the data from the old table to the new table
copy_query = " \
    INSERT INTO new_info_temp (info_id, title, people) \
    SELECT info_id, title, people \
    FROM new_info; \
"
execute_query(copy_query, conn)

# Step 3: Drop the old table
drop_old_query = " \
    DROP TABLE new_info; \
"
execute_query(drop_old_query, conn)

# Step 4: Rename the new table to the original table name
rename_query = " \
    ALTER TABLE new_info_temp \
    RENAME TO new_info; \
"
execute_query(rename_query, conn)

Query executed successfully.
Query executed successfully.
Query executed successfully.
Query executed successfully.


In [42]:
# Run the last INSERT query. Now no error is raised.

# Insert rows into the table
query = " \
	INSERT INTO new_info(title) \
	VALUES( \
	'Some New Title' \
	); \
	"
execute_query(query, conn)

Query executed successfully.


### 68. DROP Table

In [43]:
# Explore the new_info column
query = " \
	SELECT * \
	FROM new_info; \
	"
execute_query(query, conn)

   info_id           title people
0        1  Some New Title   None


In [44]:
# Drop column people

# Query in Postgre was:

#	ALTER TABLE new_info \
#	DROP COLUMN people; \

# But in SQLite it needs a series of steps, as follows:

# Step 1: Create a new table without the 'people' column
create_query = " \
    CREATE TABLE new_info_temp ( \
        info_id INTEGER PRIMARY KEY AUTOINCREMENT, \
        title TEXT NOT NULL \
    ); \
"
execute_query(create_query, conn)

# Step 2: Copy the data from the old table to the new table (excluding the 'people' column)
copy_query = " \
    INSERT INTO new_info_temp (info_id, title) \
    SELECT info_id, title \
    FROM new_info; \
"
execute_query(copy_query, conn)

# Step 3: Drop the old table
drop_old_query = " \
    DROP TABLE new_info; \
"
execute_query(drop_old_query, conn)

# Step 4: Rename the new table to the original table name
rename_query = " \
    ALTER TABLE new_info_temp \
    RENAME TO new_info; \
"
execute_query(rename_query, conn)

Query executed successfully.
Query executed successfully.
Query executed successfully.
Query executed successfully.


### 69. CHECK Constraint

In [45]:
# Create the employees table
query = " \
	CREATE TABLE employees( \
	emp_id SERIAL PRIMARY KEY, \
	first_name VARCHAR(50) NOT NULL, \
	last_name VARCHAR(50) NOT NULL, \
	birthdate DATE CHECK (birthdate > '1900-01-01'), \
	hire_date DATE CHECK (hire_date > birthdate), \
	salary INTEGER CHECK (salary > 0) \
	); \
	"
execute_query(query, conn)

Query executed successfully.


In [46]:
# Insert employee
query = " \
	INSERT INTO employees( \
	first_name, \
	last_name, \
	birthdate, \
	hire_date, \
	salary \
	) \
	VALUES( \
	'Jose', \
	'Portilla', \
	'1990-11-03', \
	'2010-01-01', \
	100 \
	); \
	"
execute_query(query, conn)
# Comment for input '1990-11-03' was /* When it was 1899-11-03, it raised an error
# But SQLite doesn't allow for comments between values

Query executed successfully.


In [47]:
# Explore the employees table
query = " \
	SELECT * \
	FROM employees; \
	"
execute_query(query, conn)

  emp_id first_name last_name   birthdate   hire_date  salary
0   None       Jose  Portilla  1990-11-03  2010-01-01     100


In [48]:
# Insert another employee
query = " \
	INSERT INTO employees( \
	first_name, \
	last_name, \
	birthdate, \
	hire_date, \
	salary \
	) \
	VALUES( \
	'Sammy', \
	'Smith', \
	'1990-11-03', \
	'2010-01-01', \
	100 \
	); \
	"
execute_query(query, conn)
# SERIAL in PK kept the count of failed attempts at inserting row!!!!

# Comment for input 100 was /* when it was -100 it raised an error
# But SQLite doesn't allow for comments between values

Query executed successfully.
